# HyperKT Objective 01 — Stage 1: Auditable data preparation
**Dataset:** ASSISTments 2009 Skill Builder. This notebook creates real, student-disjoint train/validation/test splits and training-only skill co-occurrence. It does **not** report model performance. Upload the original dataset CSV to Colab; never use synthetic observations for publication metrics.

In [ ]:
!pip -q install pandas numpy scikit-learn scipy pyarrow

In [ ]:
from pathlib import Path
import json, hashlib, platform, random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
SEED=42
random.seed(SEED); np.random.seed(SEED)
DATA_PATH='/content/skill_builder_data.csv' # CHANGE to the actual uploaded CSV
OUT=Path('/content/hyperkt_stage1'); OUT.mkdir(parents=True,exist_ok=True)
MAX_LEN=200
print('Python:',platform.python_version(),'pandas:',pd.__version__)

In [ ]:
from google.colab import files
if not Path(DATA_PATH).exists():
    print('Upload the ORIGINAL ASSISTments 2009 CSV now.')
    uploaded=files.upload()
    if len(uploaded)!=1: raise ValueError('Upload exactly one CSV')
    DATA_PATH=next(iter(uploaded))
raw_bytes=Path(DATA_PATH).read_bytes()
sha256=hashlib.sha256(raw_bytes).hexdigest()
try: df=pd.read_csv(DATA_PATH,low_memory=False)
except UnicodeDecodeError: df=pd.read_csv(DATA_PATH,encoding='latin-1',low_memory=False)
print('File SHA256:',sha256,'raw shape:',df.shape)
print('Columns:',list(df.columns))
display(df.head(3))

In [ ]:
def choose(names,required=True):
    lookup={c.lower().strip():c for c in df.columns}
    for x in names:
        if x.lower() in lookup:return lookup[x.lower()]
    if required: raise KeyError(f'Expected one of {names}; actual columns: {list(df.columns)}')
    return None
user_col=choose(['user_id','student_id','user'])
skill_col=choose(['skill_id','skill_name','kc(default)','kc(subskills)'])
correct_col=choose(['correct','correctness','is_correct'])
order_col=choose(['order_id','timestamp','start_time','problem_log_id','row'])
print(dict(user=user_col,skill=skill_col,correct=correct_col,order=order_col))

In [ ]:
data=df[[user_col,skill_col,correct_col,order_col]].copy()
data.columns=['user','skill','correct','order']
before=len(data)
data=data.dropna(subset=['user','skill','correct','order']).copy()
data['correct']=pd.to_numeric(data['correct'],errors='coerce')
data=data[data.correct.isin([0,1])].copy()
# A skill list containing ~~ or comma denotes a multi-skill event; do not silently treat it as one skill.
multiskill=data.skill.astype(str).str.contains(r'~~|,',regex=True).sum()
if multiskill: raise ValueError(f'{multiskill} multi-skill rows detected. Choose and document an explicit multi-skill policy before continuing.')
data['user']=data.user.astype(str)
data['skill']=data.skill.astype(str)
# Preserve stable original row position to break timestamp/order ties.
data['_row']=np.arange(len(data))
if order_col.lower() in ['timestamp','start_time']:
    parsed=pd.to_datetime(data.order,errors='coerce',utc=True)
    if parsed.notna().mean()<0.95: raise ValueError('Too many invalid timestamps; inspect format')
    data['order']=parsed
else:
    data['order']=pd.to_numeric(data.order,errors='coerce')
data=data.dropna(subset=['order']).sort_values(['user','order','_row'],kind='mergesort').reset_index(drop=True)
print('Retained',len(data),'of',before,'rows; students',data.user.nunique(),'skills',data.skill.nunique())
print('Class counts:',data.correct.value_counts().to_dict())

In [ ]:
# Student-disjoint 80/10/10. Split BEFORE fitting vocabulary or co-occurrence.
users=np.array(sorted(data.user.unique()))
if len(users)<20:raise ValueError('Insufficient students for a meaningful split')
train_users,holdout=train_test_split(users,test_size=.20,random_state=SEED,shuffle=True)
val_users,test_users=train_test_split(holdout,test_size=.50,random_state=SEED,shuffle=True)
sets={'train':set(train_users),'val':set(val_users),'test':set(test_users)}
assert not (sets['train']&sets['val'] or sets['train']&sets['test'] or sets['val']&sets['test'])
train=data[data.user.isin(sets['train'])].copy()
# Reserve ID=0 for unseen skill; ID=1 for padding (different IDs to make unknown handling explicit).
skills=sorted(train.skill.unique())
vocab={s:i+2 for i,s in enumerate(skills)}
for name,group in [('train',train),('val',data[data.user.isin(sets['val'])]),('test',data[data.user.isin(sets['test'])])]:
    group=group.copy();group['skill_id']=group.skill.map(vocab).fillna(0).astype('int32')
    group.to_parquet(OUT/f'{name}_interactions.parquet',index=False)
    print(name,'students',group.user.nunique(),'interactions',len(group),'unseen skill events',int((group.skill_id==0).sum()))
(OUT/'skill_vocab.json').write_text(json.dumps(vocab,indent=2))

In [ ]:
# Train-only co-occurrence: count each unordered skill pair at most once per sliding window.
from collections import Counter
counts=Counter(); window_count=0; W=5
for _,g in train.groupby('user',sort=False):
    seq=[vocab[s] for s in g.skill]
    for i in range(len(seq)):
        window=set(seq[max(0,i-W+1):i+1]); window_count+=1
        for a in window:
            for b in window:
                if a<=b: counts[(a,b)]+=1
from scipy.sparse import coo_matrix,save_npz
rr=[];cc=[];vv=[]
for (a,b),v in counts.items():
    rr.append(a);cc.append(b);vv.append(v/window_count)
    if a!=b:rr.append(b);cc.append(a);vv.append(v/window_count)
cooc=coo_matrix((vv,(rr,cc)),shape=(len(vocab)+2,len(vocab)+2)).tocsr()
save_npz(OUT/'train_cooccurrence.npz',cooc)
print('Matrix',cooc.shape,'nonzero',cooc.nnz,'windows',window_count)

In [ ]:
# Save exact split IDs and data audit. No metrics are claimed at this stage.
manifest={'seed':SEED,'source_file':Path(DATA_PATH).name,'source_sha256':sha256,'raw_rows':before,'clean_rows':len(data),'num_students':int(data.user.nunique()),'num_skills_all':int(data.skill.nunique()),'num_skills_train':len(vocab),'max_len_planned':MAX_LEN,'cooccurrence_window':W,'cooccurrence_windows':window_count,'columns':{'user':user_col,'skill':skill_col,'correct':correct_col,'order':order_col},'splits':{k:sorted(v) for k,v in sets.items()},'split_rule':'student-disjoint 80/10/10, seed 42; skill vocabulary fit on train only'}
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2,default=str))
import shutil
archive=shutil.make_archive('/content/hyperkt_stage1','zip',OUT)
print('Saved:',archive)
from google.colab import files
files.download(archive)

## Publication checkpoint
Record actual retained rows, student counts, skill counts, class distribution, missingness, unseen skills, and SHA-256 in the manuscript. **Do not use any AUC or ECE figures from the manuscript until independently reproduced.** Next: sequence construction and leakage-safe next-response prediction, then the HyperKT model.